In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
"""
Experiment 2 — ZK Proof of Communication (fixed)

Three root fixes vs previous version:
  1. Receiver actually uses candidate objects (dot-product attention over 
     the candidate set), so the referential task is genuinely solved by 
     communication, not memorisation.
  2. CIC is measured on continuous logits/vectors, not on near-one-hot 
     softmax distributions (which give KL≈0 at 100% accuracy).
  3. RC is measured by checking whether M2's token distribution shifts 
     when the query shifts — using Jensen-Shannon divergence on the logit 
     distribution, not just argmax agreement.
"""

import os, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = "cpu"
os.makedirs("results", exist_ok=True)

# ── Hyperparams ───────────────────────────────────────────────────────────────
N_FEATURES    = 16
N_DISTRACTORS = 4     # receiver picks 1 from 5
N_TRAIN       = 8000
N_VAL         = 2000
VOCAB_SIZE    = 16
QUERY_DIM     = 16
HIDDEN        = 64
N_EPOCHS      = 60
BATCH_SIZE    = 128
LR            = 1e-3
GS_TEMP       = 0.5
N_SCRAMBLES   = 30

# ── Data ──────────────────────────────────────────────────────────────────────
def make_dataset(n_samples, seed=42):
    rng = np.random.default_rng(seed)
    objects = rng.integers(0, 2,
        (n_samples, N_DISTRACTORS+1, N_FEATURES)).astype(np.float32)
    sender_input   = torch.tensor(objects[:, 0, :])
    labels         = torch.zeros(n_samples, dtype=torch.long)
    receiver_input = torch.tensor(objects)
    return TensorDataset(sender_input, labels, receiver_input)

train_ds = make_dataset(N_TRAIN, seed=42)
val_ds   = make_dataset(N_VAL,   seed=99)

# ── Agents (receiver NOW uses candidate objects) ──────────────────────────────
class Sender(nn.Module):
    def __init__(self):
        super().__init__()
        self.turn1 = nn.Sequential(
            nn.Linear(N_FEATURES, HIDDEN), nn.ReLU(),
            nn.Linear(HIDDEN, VOCAB_SIZE)
        )
        self.turn2 = nn.Sequential(
            nn.Linear(N_FEATURES + QUERY_DIM, HIDDEN), nn.ReLU(),
            nn.Linear(HIDDEN, VOCAB_SIZE)
        )

    def forward_turn1(self, obj):
        return self.turn1(obj)                              # (B, VOCAB_SIZE)

    def forward_turn2(self, obj, query):
        return self.turn2(torch.cat([obj, query], dim=-1)) # (B, VOCAB_SIZE)


class Receiver(nn.Module):
    """
    Uses dot-product attention between message embedding and each candidate
    object embedding. This is the fix: receiver_input is actually used.
    """
    def __init__(self):
        super().__init__()
        self.msg_proj  = nn.Linear(VOCAB_SIZE, HIDDEN)
        self.obj_proj  = nn.Linear(N_FEATURES, HIDDEN)
        self.query_mlp = nn.Sequential(
            nn.Linear(HIDDEN, HIDDEN), nn.ReLU(),
            nn.Linear(HIDDEN, QUERY_DIM), nn.Tanh()
        )

    def _attend(self, msg_h, receiver_input):
        """msg_h: (B,H)  receiver_input: (B, n_cand, N_FEATURES) → (B, n_cand)"""
        obj_h  = self.obj_proj(receiver_input)              # (B, n_cand, H)
        scores = torch.bmm(obj_h, msg_h.unsqueeze(-1)).squeeze(-1)  # (B, n_cand)
        return scores

    def forward_turn1(self, msg, receiver_input):
        """Returns query vector AND turn-1 candidate scores."""
        msg_h  = self.msg_proj(msg)                         # (B, H)
        query  = self.query_mlp(msg_h)                      # (B, QUERY_DIM)
        scores = self._attend(msg_h, receiver_input)        # (B, n_cand)
        return query, scores

    def forward_turn2(self, msg1, msg2, receiver_input):
        """Returns final candidate scores using both messages."""
        h1     = self.msg_proj(msg1)
        h2     = self.msg_proj(msg2)
        # average both message representations, attend over candidates
        scores = self._attend((h1 + h2) / 2, receiver_input)
        return scores                                        # (B, n_cand)


# ── Game wrapper ──────────────────────────────────────────────────────────────
class MultiTurnGame(nn.Module):
    def __init__(self, sender, receiver, severed=False):
        super().__init__()
        self.sender   = sender
        self.receiver = receiver
        self.severed  = severed  # if True: receiver never sees real messages

    def forward(self, si, labels, ri, tau=GS_TEMP):
        B = si.size(0)

        # Turn 1
        logits1 = self.sender.forward_turn1(si)
        if self.severed:
            msg1 = torch.zeros(B, VOCAB_SIZE)
        elif self.training:
            msg1 = F.gumbel_softmax(logits1, tau=tau, hard=True)
        else:
            msg1 = F.one_hot(logits1.argmax(-1), VOCAB_SIZE).float()

        query, scores1 = self.receiver.forward_turn1(msg1, ri)

        # Turn 2
        logits2 = self.sender.forward_turn2(si, query)
        if self.severed:
            msg2 = torch.zeros(B, VOCAB_SIZE)
        elif self.training:
            msg2 = F.gumbel_softmax(logits2, tau=tau, hard=True)
        else:
            msg2 = F.one_hot(logits2.argmax(-1), VOCAB_SIZE).float()

        scores2 = self.receiver.forward_turn2(msg1, msg2, ri)

        loss = F.cross_entropy(scores2, labels)
        acc  = (scores2.argmax(-1) == labels).float().mean()
        return loss, acc, logits1, logits2, msg1, msg2, query, scores1, scores2


# ── Training ──────────────────────────────────────────────────────────────────
def train_game(severed=False, label=""):
    sender   = Sender()
    receiver = Receiver()
    game     = MultiTurnGame(sender, receiver, severed=severed)
    opt      = torch.optim.Adam(game.parameters(), lr=LR)

    loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    for epoch in range(N_EPOCHS):
        game.train()
        for si, lb, ri in loader:
            opt.zero_grad()
            loss, *_ = game(si, lb, ri)
            loss.backward()
            opt.step()

        if (epoch+1) % 10 == 0:
            game.eval()
            accs = []
            with torch.no_grad():
                for si, lb, ri in DataLoader(val_ds, batch_size=256):
                    _, acc, *_ = game(si, lb, ri)
                    accs.append(acc.item())
            print(json.dumps({"label": label, "epoch": epoch+1,
                              "val_acc": round(float(np.mean(accs)), 4)}))
    return game


# ── ZK Metric ─────────────────────────────────────────────────────────────────
def js_divergence(p_logits, q_logits):
    """Jensen-Shannon divergence between two logit distributions. (B,) → scalar"""
    p = F.softmax(p_logits, dim=-1)
    q = F.softmax(q_logits, dim=-1)
    m = 0.5 * (p + q)
    js = 0.5 * (p * (p / m.clamp(1e-9)).log()).sum(-1) + \
         0.5 * (q * (q / m.clamp(1e-9)).log()).sum(-1)
    return js.mean().item()


def compute_zk_metric(game, n_scrambles=N_SCRAMBLES):
    """
    Three quantities, all content-agnostic:

    CIC_turn1: how much does scrambling M1 shift the query vector?
               measured as mean L2 distance between real and scrambled queries.
               Nonzero iff receiver is actually using M1.

    RC:        how much does the shifted query change M2's token distribution?
               measured as mean JS divergence between logits2(real_query)
               and logits2(scrambled_query).
               Nonzero iff sender is actually using the query.

    CIC_turn2: how much does scrambling M2 shift the final candidate scores?
               measured as mean L2 distance on final logit vectors.
               Nonzero iff receiver is actually using M2.

    ZK score = mean(CIC1, CIC2) * RC
    All three must be nonzero for ZK > 0.
    """
    game.eval()
    loader = DataLoader(val_ds, batch_size=256, shuffle=False)

    cic1_all, rc_all, cic2_all = [], [], []

    with torch.no_grad():
        for si, lb, ri in loader:
            B = si.size(0)

            # ── Real forward pass ─────────────────────────────────────────
            logits1_real = game.sender.forward_turn1(si)
            msg1_real    = F.one_hot(logits1_real.argmax(-1), VOCAB_SIZE).float()
            query_real, _ = game.receiver.forward_turn1(msg1_real, ri)
            logits2_real  = game.sender.forward_turn2(si, query_real)
            msg2_real     = F.one_hot(logits2_real.argmax(-1), VOCAB_SIZE).float()
            scores2_real  = game.receiver.forward_turn2(msg1_real, msg2_real, ri)

            for _ in range(n_scrambles):
                perm = torch.randperm(B)

                # ── CIC1: scramble M1 → measure query shift ───────────────
                msg1_fake      = msg1_real[perm]
                query_fake, _  = game.receiver.forward_turn1(msg1_fake, ri)
                delta_query    = (query_real - query_fake).pow(2).sum(-1).sqrt()
                cic1_all.append(delta_query.mean().item())

                # ── RC: shifted query → shift in M2 logit distribution ────
                logits2_fake   = game.sender.forward_turn2(si, query_fake)
                rc_js          = js_divergence(logits2_real, logits2_fake)
                rc_all.append(rc_js)

                # ── CIC2: scramble M2 → measure final score shift ─────────
                msg2_fake      = msg2_real[perm]
                scores2_fake   = game.receiver.forward_turn2(msg1_real, msg2_fake, ri)
                delta_scores   = (scores2_real - scores2_fake).pow(2).sum(-1).sqrt()
                cic2_all.append(delta_scores.mean().item())

    cic1 = float(np.mean(cic1_all))
    rc   = float(np.mean(rc_all))
    cic2 = float(np.mean(cic2_all))
    zk   = ((cic1 + cic2) / 2) * rc

    return {
        "cic_turn1": round(cic1, 5),
        "rc_coeff":  round(rc,   5),
        "cic_turn2": round(cic2, 5),
        "zk_score":  round(zk,   5),
    }


# ── Run ───────────────────────────────────────────────────────────────────────
print("=== Training communicating game ===")
comm_game = train_game(severed=False, label="communicating")

print("\n=== Training severed-channel baseline ===")
sev_game  = train_game(severed=True,  label="severed")

print("\n=== Computing ZK metric ===")
comm_scores = compute_zk_metric(comm_game)
sev_scores  = compute_zk_metric(sev_game)

print("\nCommunicating game:")
print(json.dumps(comm_scores, indent=2))
print("\nSevered baseline:")
print(json.dumps(sev_scores, indent=2))

results = {"communicating": comm_scores, "severed_baseline": sev_scores}
with open("results/zk_results.json", "w") as f:
    json.dump(results, f, indent=2)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

metrics = ["cic_turn1", "rc_coeff", "cic_turn2", "zk_score"]
colors  = ["#378ADD", "#1D9E75", "#EF9F27", "#E24B4A"]

x = np.arange(len(metrics))
w = 0.3
axes[0].bar(x - w/2, [comm_scores[m] for m in metrics], w,
            color=colors, label="communicating", alpha=0.9)
axes[0].bar(x + w/2, [sev_scores[m]  for m in metrics], w,
            color=colors, label="severed", alpha=0.4)
axes[0].set_xticks(x)
axes[0].set_xticklabels(["CIC turn1\n(query shift)", "RC\n(M2 responds\nto query)",
                          "CIC turn2\n(score shift)", "ZK score\n(product)"])
axes[0].set_title("ZK metric: communicating vs severed")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

# Ratio plot
ratios = [(comm_scores[m]+1e-9) / (sev_scores[m]+1e-9) for m in metrics]
bar_colors = ["#1D9E75" if r > 1 else "#E24B4A" for r in ratios]
axes[1].bar(metrics, ratios, color=bar_colors)
axes[1].axhline(y=1, color="gray", linestyle="--", alpha=0.7)
axes[1].set_title("Communicating / severed ratio\n(>1 means communication detected)")
axes[1].set_yscale("log")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("results/zk_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

ratio = (comm_scores["zk_score"]+1e-9) / (sev_scores["zk_score"]+1e-9)
print(f"\nZK ratio (communicating / baseline): {ratio:.2f}x")
print("A ratio >> 1 confirms genuine recursive causal communication.")